# Kimi-Linear (GDN-2) — training on Kaggle **2× NVIDIA T4** (data-parallel)

This notebook pretrains and instruction-tunes a ~140M Kimi-Linear (GDN-2) code LM using
**both** T4 GPUs via pure data parallelism (model replicated per GPU, batch split across them —
see `training/parallel.py`). The math is identical to a single-device run of the same global batch.

**Before running:** Settings → Accelerator = **GPU T4 ×2**, and **Internet = ON** (needed for the
one-time HuggingFace tokenization/streaming).

Notes:
- Precision is **float32** — T4 (Turing) has no hardware bf16.
- `batch_size` must stay a **multiple of 2** (the device count); the trainer exits clearly otherwise.
- Checkpoints go under **`/kaggle/working`** so they survive the session (resume with `--resume`).


## 1. Check the GPUs


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 2. Clone the repo & install the JAX stack
Replace `REPO_URL` with your fork if needed.


In [ ]:
REPO_URL = 'https://github.com/wisnunugroho21/nugie-coding-llm-agent-3.git'
import os
if not os.path.isdir('nugie-coding-llm-agent-3'):
    !git clone $REPO_URL
%cd /kaggle/working/nugie-coding-llm-agent-3

In [ ]:
# Kaggle preinstalls JAX; upgrading only the Python `jax` leaves an OLDER
# compiled `jaxlib` and triggers `register_attribute_builder ... allow_existing`.
# Uninstall the whole JAX family first, then install a MATCHED cuda12 set.
!pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt
!pip install -q -U "jax[cuda12]" flax optax orbax-checkpoint grain datasets tokenizers pyyaml
# The two Version: lines MUST match (e.g. both 0.10.x):
!pip show jax jaxlib 2>/dev/null | grep -E '^(Name|Version)'

## ⚠️ 2b. RESTART THE KERNEL after installing
The old `jaxlib` is already loaded in this process, so the reinstall only takes effect after a restart.
Run the cell below (or **Run → Restart & clear cell outputs**), then continue at step 3.
**Do NOT `import jax` before restarting.**

In [ ]:
# Restarts the Python kernel so the freshly installed jaxlib is the one that loads.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## 3. Confirm JAX sees **both** T4s
This must print **2** devices, or training will use only one GPU.


In [ ]:
# After the restart the working dir resets — re-enter the repo.
%cd /kaggle/working/nugie-coding-llm-agent-3
import jax
print('backend:', jax.default_backend())
print('devices:', jax.devices())
assert jax.device_count() == 2, 'Expected 2 T4s — set Accelerator to GPU T4 x2.'

## 4. Send checkpoints to persistent storage
Rewrite the config `out_dir`s to live under `/kaggle/working` so they persist across the 9–12h session.


In [ ]:
!sed -i 's#runs/t4_pretrain#/kaggle/working/runs/t4_pretrain#g' configs/t4_pretrain.yaml configs/t4_sft.yaml
!sed -i 's#runs/t4_sft#/kaggle/working/runs/t4_sft#g' configs/t4_sft.yaml
!grep -E 'out_dir|tokenizer_path' configs/t4_pretrain.yaml configs/t4_sft.yaml

## 5. Pretrain (uses both T4s)
First launch tokenizes the corpus once into `.npy` shards (needs Internet), then streams them.
Look for a `data-parallel over 2 devices (8 rows/device per step)` log line.

Session about to time out? Re-run this cell with `--resume` appended — it restores weights,
optimizer, and the exact data position.


In [ ]:
!python -m training.train --config configs/t4_pretrain.yaml

In [ ]:
# Resume an interrupted pretrain run (safe to re-run):
# !python -m training.train --config configs/t4_pretrain.yaml --resume

## 6. Instruction-tune (SFT), warm-started from the pretrained weights
`--init-from` loads the pretrain run's **weights only** (fresh optimizer/data, step 0); the SFT
config reuses the pretrain tokenizer so the vocab matches.


In [ ]:
!python -m training.train --config configs/t4_sft.yaml --init-from /kaggle/working/runs/t4_pretrain

## 7. Evaluate — perplexity + sample generations


In [ ]:
!python -m training.evaluate --config configs/t4_sft.yaml

## 8. (Optional) Functional HumanEval pass@1
⚠️ This **executes model-generated code**. Kaggle runs each program in a subprocess with a timeout,
but treat the session as disposable. Start with a small `--humaneval-limit`.


In [ ]:
# !python -m training.evaluate --config configs/t4_sft.yaml --humaneval --humaneval-limit 20

---
### Tips
- **OOM?** Lower `train.batch_size` to 8 (keep it a multiple of 2) or `data.seq_len` to 512 in the YAML.
- **Only 1 GPU shows up?** The code still runs (replicate/split become no-ops); re-check the accelerator setting for both.
- This is **data parallelism** — it needs the model to fit on one 16 GB T4 (this config does). The 4B `h200` config
  needs parameter sharding (FSDP), see `docs/multi_device_plan.md`.
